# 📥 Terminal 2 — Output Consumer
Reads from the `predictions` topic and prints each result live to the console.

**Run this SECOND** (after the Processor is running, before the Producer).

In [ ]:
!pip install confluent-kafka --quiet

In [ ]:
# ════════════════════════════════════════════════
#   FILL IN YOUR CONFLUENT CLOUD CREDENTIALS HERE
# ════════════════════════════════════════════════
BOOTSTRAP_SERVERS = 'pkc-XXXXX.us-east-1.aws.confluent.cloud:9092'  # <-- replace
SASL_USERNAME     = 'YOUR_API_KEY'                                    # <-- replace
SASL_PASSWORD     = 'YOUR_API_SECRET'                                 # <-- replace
PREDICTIONS_TOPIC = 'predictions'
# ════════════════════════════════════════════════

In [ ]:
import json
import time
from confluent_kafka import Consumer, KafkaException

conf = {
    'bootstrap.servers':  BOOTSTRAP_SERVERS,
    'security.protocol':  'SASL_SSL',
    'sasl.mechanisms':    'PLAIN',
    'sasl.username':      SASL_USERNAME,
    'sasl.password':      SASL_PASSWORD,
    'group.id':           'output-consumer-group',
    'auto.offset.reset':  'earliest',
}

consumer = Consumer(conf)
consumer.subscribe([PREDICTIONS_TOPIC])
print(f'Consumer subscribed to [{PREDICTIONS_TOPIC}]. Waiting for predictions...\n')
print('=' * 70)

WEATHER = {1: 'Clear', 2: 'Cloudy', 3: 'Light Rain', 4: 'Heavy Rain'}
SEASON  = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}

try:
    while True:
        msg = consumer.poll(timeout=2.0)

        if msg is None:
            print('[WAITING] No messages yet...', end='\r')
            continue

        if msg.error():
            raise KafkaException(msg.error())

        data = json.loads(msg.value().decode('utf-8'))

        predicted = data.get('predicted_count', '?')
        actual    = data.get('actual_count', '?')
        row       = data.get('row_index', '?')
        hr        = data.get('hr', '?')
        temp_raw  = data.get('temp', 0)
        temp_c    = round(temp_raw * 41)   # normalised → Celsius (dataset uses 0–1 scale, max=41°C)
        weather   = WEATHER.get(int(data.get('weathersit', 1)), 'Unknown')
        season    = SEASON.get(int(data.get('season', 1)), 'Unknown')

        print(f"\n🚲 PREDICTION #{row}")
        print(f"   Hour     : {hr}:00  |  Season: {season}  |  Weather: {weather}  |  Temp: {temp_c}°C")
        print(f"   Predicted Rentals : {predicted}")
        print(f"   Actual Rentals    : {actual}")
        if actual and actual != '?':
            diff = abs(float(predicted) - float(actual))
            print(f"   Difference        : {diff:.1f} bikes")
        print('─' * 70)

except KeyboardInterrupt:
    print('\n[STOPPED] Consumer shut down.')
finally:
    consumer.close()